### **Configuração Pasta Raiz**

In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Adiciona a raiz do projeto ao path do Python para que as importações funcionem
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# print(f"Raiz do projeto definida como: {PROJECT_ROOT}")

### **Bibliotecas**

In [2]:
import pandas as pd
import src.utils.converters as convert
from src.config import FILES

### **Leitura da base de dados**

In [4]:
dfs = {}

for key, filename in FILES.items():
    file_path = f'../{filename}'
    
    try:
        dfs[key] = pd.read_csv(file_path)
        
        print(f"✅ {key.ljust(15)}: Carregado ({len(dfs[key])} linhas)")
        
    except FileNotFoundError:
        print(f"❌ {key.ljust(15)}: Arquivo não encontrado em '{file_path}'")
    except Exception as e:
        print(f"❌ {key.ljust(15)}: Erro - {e}")

✅ draft          : Carregado (60 linhas)
✅ nba            : Carregado (299 linhas)
✅ college_adv    : Carregado (118 linhas)
✅ college_per_min: Carregado (118 linhas)
✅ college_player_metadata: Carregado (51 linhas)
✅ mock           : Carregado (60 linhas)
✅ combine_anthro : Carregado (69 linhas)
✅ combine_strength_agility: Carregado (69 linhas)


### **Executando cortes nas bases**

#### **Draft**

In [5]:
dfs['draft'].head()

,Rk,Pk,Tm,Player,College,Yrs,G,MP,PTS,TRB,...,RPG,APG,WS,WS_per_48,BPM,VORP,Year,Player_Link,Player_ID,College_Link
0,1,1,PHO,Deandre Ayton,Arizona,8.0,412.0,12697.0,6773.0,4290.0,...,10.4,1.6,38.7,0.146,0.7,8.6,2018,https://www.basketball-reference.com/players/a...,aytonde01,https://www.sports-reference.com/cbb/players/d...
1,2,2,SAC,Marvin Bagley III,Duke,8.0,303.0,6695.0,3586.0,1951.0,...,6.4,0.9,13.1,0.094,-1.5,0.7,2018,https://www.basketball-reference.com/players/b...,baglema01,https://www.sports-reference.com/cbb/players/m...
2,3,3,ATL,Luka Dončić,NaN,8.0,461.0,16117.0,13259.0,3975.0,...,8.6,8.2,59.1,0.176,7.7,39.4,2018,https://www.basketball-reference.com/players/d...,doncilu01,NaN
3,4,4,MEM,Jaren Jackson Jr.,Michigan State,8.0,422.0,12074.0,7786.0,2337.0,...,5.5,1.5,29.3,0.116,0.8,8.7,2018,https://www.basketball-reference.com/players/j...,jacksja02,https://www.sports-reference.com/cbb/players/j...
4,5,5,DAL,Trae Young,Oklahoma,8.0,488.0,16761.0,12309.0,1697.0,...,3.5,9.8,43.5,0.125,2.6,19.3,2018,https://www.basketball-reference.com/players/y...,youngtr01,https://www.sports-reference.com/cbb/players/t...


In [6]:
df_draft = dfs['draft'][['Player_ID','Player','Year','College','Pk']].copy()

In [7]:
df_draft.rename(columns={
    'Player': 'Player_Name', 
    'Pk': 'Draft_Pick',
    'Year': 'Draft_Year',
    'College': 'College_Name'
}, inplace=True)

In [8]:
df_draft.head()

,Player_ID,Player_Name,Draft_Year,College_Name,Draft_Pick
0,aytonde01,Deandre Ayton,2018,Arizona,1
1,baglema01,Marvin Bagley III,2018,Duke,2
2,doncilu01,Luka Dončić,2018,NaN,3
3,jacksja02,Jaren Jackson Jr.,2018,Michigan State,4
4,youngtr01,Trae Young,2018,Oklahoma,5


#### **Estatísticas Avançadas da NBA**

In [9]:
dfs['nba'].head()

,Season,Age,Team,Lg,Pos,G,GS,MP,PER,TS_pct,...,OWS,DWS,WS,WS_per_48,OBPM,DBPM,BPM,VORP,Awards,Player_ID
0,2018-19,20,PHO,NBA,C,71.0,70.0,2183.0,20.5,0.608,...,4.1,1.7,5.8,0.128,1.3,-1.1,0.2,1.2,ROY-3,aytonde01
1,2019-20,21,PHO,NBA,C,38.0,32.0,1236.0,20.0,0.568,...,1.5,1.6,3.1,0.121,0.8,-0.4,0.4,0.7,NaN,aytonde01
2,2020-21,22,PHO,NBA,C,69.0,69.0,2115.0,20.3,0.653,...,5.3,3.1,8.4,0.191,1.1,0.1,1.2,1.7,NaN,aytonde01
3,2021-22,23,PHO,NBA,C,58.0,58.0,1713.0,21.9,0.656,...,4.1,3.1,7.3,0.203,2.2,0.6,2.8,2.1,NaN,aytonde01
4,2022-23,24,PHO,NBA,C,67.0,67.0,2035.0,19.9,0.617,...,3.3,3.0,6.2,0.147,0.9,0.0,0.9,1.5,NaN,aytonde01


In [10]:
df_nba_stats = dfs['nba'][['Player_ID','Age','VORP','Season']].copy()

In [11]:
df_nba_stats.head()

,Player_ID,Age,VORP,Season
0,aytonde01,20,1.2,2018-19
1,aytonde01,21,0.7,2019-20
2,aytonde01,22,1.7,2020-21
3,aytonde01,23,2.1,2021-22
4,aytonde01,24,1.5,2022-23


##### **Aplicação de limpeza e filtros**

In [12]:
# 1. Classificar e Filtrar: Queremos apenas as primeiras 5 temporadas de cada jogador.
# A forma mais simples de fazer isso é agrupar, ordenar e pegar o head(5)

df_nba_stats['Season_Int'] = df_nba_stats['Season'].str.split('-').str[0].astype(int)

In [13]:
# Agrupa por jogador, ordena por ano, e pega apenas as 5 primeiras linhas
first_five_years = (
    df_nba_stats
    .sort_values(by=['Player_ID', 'Season_Int'])
    .groupby('Player_ID')
    .head(5)
)

In [14]:
# 2. Calcular o VORP Médio (Métrica Alvo)
# Calcular a soma total de VORP nos 5 primeiros anos
df_nba_stats = (
    first_five_years
    .groupby('Player_ID')
    .agg(
        VORP_5Y_SUM=('VORP', 'sum'),  # Soma o VORP dos 5 anos
        Rookie_Age=('Age', 'min'),     # Pega a menor idade (Idade de estreia)
        Seasons_Played=('Season', 'count') # Conta quantas temporadas ele jogou
    )
    .reset_index()
)


In [15]:
df_nba_stats.sort_values(by='VORP_5Y_SUM', ascending=False)

,Player_ID,VORP_5Y_SUM,Rookie_Age,Seasons_Played
48,youngtr01,15.2,20,5
18,gilgesh01,13.1,20,5
6,bridgmi01,10.3,22,5
47,williro04,7.5,21,5
2,aytonde01,7.2,20,5
10,brunsja01,7.0,22,5
25,jacksja02,5.0,19,5
29,meltode01,4.6,20,5
34,portemi01,4.2,21,5
7,bridgmi02,3.8,20,5


#### **Estatísticas Avançadas do College**

In [16]:
dfs['college_adv'].head()

,Season,Tm,Conf,Class,Pos,G,GS,MP,PER,TS_pct,...,USG_pct,OWS,DWS,WS,WS_per_40,OBPM,DBPM,BPM,Awards,Player_ID
0,2017-18,Arizona,Pac-12,FR,F,35.0,35.0,1172.0,32.6,0.650,...,26.6,5.5,2.1,7.6,0.259,8.6,2.4,11.0,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",aytonde01
1,2017-18,Duke,ACC,FR,F,33.0,32.0,1118.0,30.6,0.643,...,26.3,4.8,2.1,6.9,0.248,8.8,2.0,10.8,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",baglema01
2,2017-18,Michigan State,Big Ten,FR,F,35.0,34.0,764.0,26.9,0.647,...,23.5,2.3,2.2,4.6,0.239,4.4,7.3,11.7,NaN,jacksja02
3,2017-18,Oklahoma,Big 12,FR,G,32.0,32.0,1133.0,28.3,0.585,...,37.1,4.6,1.1,5.7,0.201,9.7,1.3,11.1,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",youngtr01
4,2017-18,Texas,Big 12,FR,F,30.0,29.0,906.0,28.3,0.593,...,21.3,2.4,2.4,4.8,0.212,5.0,5.9,10.9,NaN,bambamo01


In [17]:
df_college_adv_init = dfs['college_adv'][['Player_ID','Season','TS_pct','3PAr','FTr','USG_pct','OWS','DWS','WS_per_40']].copy()

In [18]:
df_college_adv_init.rename(columns={
    'TS_pct': 'true_shooting_pct', 
    '3PAr': 'three_point_attempt_rate',
    'FTr': 'free_throw_attempt_rate',
    'USG_pct': 'usage_pct',
    'OWS': 'offensive_win_shares',
    'DWS': 'defensive_win_shares',
    'WS_per_40': 'win_shares_per_40min'
}, inplace=True)

In [19]:
df_college_adv_init.head()

,Player_ID,Season,true_shooting_pct,three_point_attempt_rate,free_throw_attempt_rate,usage_pct,offensive_win_shares,defensive_win_shares,win_shares_per_40min
0,aytonde01,2017-18,0.650,0.078,0.424,26.6,5.5,2.1,0.259
1,baglema01,2017-18,0.643,0.132,0.475,26.3,4.8,2.1,0.248
2,jacksja02,2017-18,0.647,0.414,0.573,23.5,2.3,2.2,0.239
3,youngtr01,2017-18,0.585,0.531,0.443,37.1,4.6,1.1,0.201
4,bambamo01,2017-18,0.593,0.189,0.441,21.3,2.4,2.4,0.212


##### **Aplicação de limpeza e filtros**

In [20]:
# 1. Ordenar: Primeiro por Jogador, depois por Temporada (do mais antigo para o mais novo)
df_sorted = df_college_adv_init.sort_values(by=['Player_ID', 'Season'], ascending=[True, True])

# 2. Filtrar: Remove duplicatas de ID, mantendo APENAS a última linha (a temporada mais recente)
df_college_adv = df_sorted.drop_duplicates(subset=['Player_ID'], keep='last').copy()

# 3. Drop Column: Remove coluna 'Season' que não é mais necessária
df_college_adv.drop('Season', axis=1, inplace=True)

In [21]:
df_college_adv.head()

,Player_ID,true_shooting_pct,three_point_attempt_rate,free_throw_attempt_rate,usage_pct,offensive_win_shares,defensive_win_shares,win_shares_per_40min
31,allengr01,0.580,0.637,0.293,20.7,4.0,2.0,0.179
117,antetko01,0.575,0.149,0.634,20.9,0.2,0.5,0.062
0,aytonde01,0.650,0.078,0.424,26.6,5.5,2.1,0.259
1,baglema01,0.643,0.132,0.475,26.3,4.8,2.1,0.248
4,bambamo01,0.593,0.189,0.441,21.3,2.4,2.4,0.212


#### **Estatísticas Game per Min do College**

In [22]:
dfs['college_per_min']

,Season,Tm,Conf,Class,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Player_ID
0,2017-18,Arizona,Pac-12,FR,F,35.0,35.0,1172.0,9.4,15.4,...,9.8,13.8,1.9,0.7,2.3,2.4,2.8,24.0,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",aytonde01
1,2017-18,Duke,ACC,FR,F,33.0,32.0,1118.0,9.7,15.7,...,8.4,13.1,1.8,1.0,1.0,2.7,2.1,24.8,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",baglema01
2,2017-18,Michigan State,Big Ten,FR,F,35.0,34.0,764.0,6.2,12.1,...,7.9,10.6,2.0,1.1,5.5,3.2,5.9,20.0,NaN,jacksja02
3,2017-18,Oklahoma,Big 12,FR,G,32.0,32.0,1133.0,9.2,21.8,...,3.9,4.4,9.8,1.9,0.3,5.9,2.0,30.9,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",youngtr01
4,2017-18,Texas,Big 12,FR,F,30.0,29.0,906.0,6.4,11.9,...,9.7,14.0,0.7,1.0,4.9,2.0,3.4,17.1,NaN,bambamo01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,2013-14,Colorado,Pac-12,FR,G,27.0,0.0,148.0,3.0,10.5,...,9.7,11.1,1.4,1.1,0.3,3.2,3.8,11.1,NaN,kingge03
114,2015-16,Colorado,Pac-12,SO,G,34.0,27.0,862.0,7.1,15.8,...,4.8,7.4,1.1,0.6,0.4,2.9,2.8,21.4,NaN,kingge03
115,2016-17,Colorado,Pac-12,JR,G,34.0,31.0,943.0,5.9,12.8,...,6.9,9.8,1.1,0.7,0.3,2.4,3.1,16.0,NaN,kingge03
116,2017-18,Colorado,Pac-12,SR,G,32.0,31.0,913.0,6.2,14.0,...,8.5,11.0,1.5,0.7,0.9,2.8,3.4,18.1,NaN,kingge03


In [23]:
df_college_per_min_init = dfs['college_per_min'][['Player_ID','Season', 'Conf', 'Class', 'G', 'GS', 'MP',  '3P', '3PA', 'eFG_pct', 'FTA', 'FT_pct', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Awards', '3P_pct' ]]

In [24]:
df_college_per_min_init.head()

,Player_ID,Season,Conf,Class,G,GS,MP,3P,3PA,eFG_pct,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,3P_pct
0,aytonde01,2017-18,Pac-12,FR,35.0,35.0,1172.0,0.4,1.2,0.625,...,9.8,13.8,1.9,0.7,2.3,2.4,2.8,24.0,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",0.343
1,baglema01,2017-18,ACC,FR,33.0,32.0,1118.0,0.8,2.1,0.640,...,8.4,13.1,1.8,1.0,1.0,2.7,2.1,24.8,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",0.397
2,jacksja02,2017-18,Big Ten,FR,35.0,34.0,764.0,2.0,5.0,0.595,...,7.9,10.6,2.0,1.1,5.5,3.2,5.9,20.0,NaN,0.396
3,youngtr01,2017-18,Big 12,FR,32.0,32.0,1133.0,4.2,11.6,0.518,...,3.9,4.4,9.8,1.9,0.3,5.9,2.0,30.9,"AA-1,AP-AA-1,NABC-AA-1,SN-AA-1,USBWA-AA-1,W-AA-1",0.360
4,bambamo01,2017-18,Big 12,FR,30.0,29.0,906.0,0.6,2.3,0.567,...,9.7,14.0,0.7,1.0,4.9,2.0,3.4,17.1,NaN,0.275


In [25]:
# 1. Preparar o DataFrame de Carreira (Soma dos Totais)
# Agrupamos por ID e somamos apenas as colunas de "Volume"
df_career_totals = df_college_per_min_init.groupby('Player_ID').agg({
    'G': 'sum',
    'GS': 'sum',
    'MP': 'sum',
    'Awards': 'count' 
}).reset_index()

df_career_totals.rename(columns={
    'G': 'Career_Games',
    'GS': 'Career_Starts',
    'MP': 'Career_Minutes',
    'Awards': 'Career_Awards_Count'
}, inplace=True)

In [26]:
# 2. Preparar o DataFrame da Última Temporada (Habilidade Atual)
# Ordena por jogador e ano, e pega a última linha
df_last_season = df_college_per_min_init.sort_values(['Player_ID', 'Season']).drop_duplicates('Player_ID', keep='last')

cols_performance = [
    'Player_ID', 'Season', 'Conf', 'Class', # Identificadores
    '3P', '3PA', 'eFG_pct', 'FTA', 'FT_pct',             # Arremesso
    'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', '3P_pct' # Stats Puras
]
df_last_season = df_last_season[cols_performance]

In [27]:
# 3. Juntar tudo no Dataset Final do Modelo
df_college_per_min = pd.merge(df_last_season, df_career_totals, on='Player_ID', how='left')

# 4. Converter 'Class' para 'Years_College'
df_college_per_min['Years_College'] = df_college_per_min['Class'].apply(convert.class_to_int)

# 5. Drop Column: Remove coluna 'Season' que não é mais necessária
df_college_per_min.drop(['Season', 'Class'], axis=1, inplace=True)

In [28]:
df_college_per_min.rename(columns={
    'Conf': 'Conference',
    '3P': 'Three_Pointers_Made',
    '3PA': 'Three_Pointers_Attempted',
    'eFG_pct': 'Effective_Field_Goal_Pct',
    'FTA': 'Free_Throw_Attempted',
    'FT_pct': 'Free_Throw_Pct',
    'ORB': 'Offensive_Rebounds',
    'DRB': 'Defensive_Rebounds',
    'TRB': 'Total_Rebounds',
    'AST': 'Assists',
    'STL': 'Steals',
    'BLK': 'Blocks',
    'TOV': 'Turnovers',
    'PF': 'Personal_Fouls',
    'PTS': 'Points',
    '3P_pct': 'Three_Point_Pct'
}, inplace=True)

In [29]:
# Verificação
df_college_per_min.head()

,Player_ID,Conference,Three_Pointers_Made,Three_Pointers_Attempted,Effective_Field_Goal_Pct,Free_Throw_Attempted,Free_Throw_Pct,Offensive_Rebounds,Defensive_Rebounds,Total_Rebounds,...,Blocks,Turnovers,Personal_Fouls,Points,Three_Point_Pct,Career_Games,Career_Starts,Career_Minutes,Career_Awards_Count,Years_College
0,allengr01,ACC,3.1,8.4,0.536,3.9,0.850,0.6,3.1,3.7,...,0.1,2.4,2.4,17.3,0.370,142.0,97.0,3965.0,1,4
1,antetko01,A-10,0.2,1.4,0.584,5.8,0.516,2.2,5.4,7.6,...,2.8,3.6,6.0,13.8,0.133,29.0,6.0,438.0,0,1
2,aytonde01,Pac-12,0.4,1.2,0.625,6.5,0.733,4.0,9.8,13.8,...,2.3,2.4,2.8,24.0,0.343,35.0,35.0,1172.0,1,1
3,baglema01,ACC,0.8,2.1,0.640,7.5,0.627,4.7,8.4,13.1,...,1.0,2.7,2.1,24.8,0.397,33.0,32.0,1118.0,1,1
4,bambamo01,Big 12,0.6,2.3,0.567,5.3,0.681,4.2,9.7,14.0,...,4.9,2.0,3.4,17.1,0.275,30.0,29.0,906.0,0,1


#### **Mock do Draft**

In [30]:
dfs['mock'].head()

,Mock_Pick,College,Weight,Pos,Team,Height,Player,Class,Year
0,1,Arizona,260,C,Phoenix,7-0,Deandre Ayton,Fr.,2018
1,2,Duke,235,PF,Sacramento,6-11,Marvin Bagley,Fr.,2018
2,3,Slovenia,225,SG/SF,Dallas,6-8,Luka Doncic,Intl.,2018
3,4,Michigan St.,235,PF/C,Memphis,6-11,Jaren Jackson Jr.,Fr.,2018
4,5,Texas,225,C,Atlanta,7-0,Mohamed Bamba,Fr.,2018


In [31]:
df_mock_draft = dfs['mock'][['Player','Year','Mock_Pick']].copy()

In [32]:
# Renomeia para não confundir depois
df_mock_draft.rename(columns={
    'Player': 'Player_Name',
    'Year': 'Draft_Year'
}, inplace=True)

In [33]:
df_mock_draft.head()

,Player_Name,Draft_Year,Mock_Pick
0,Deandre Ayton,2018,1
1,Marvin Bagley,2018,2
2,Luka Doncic,2018,3
3,Jaren Jackson Jr.,2018,4
4,Mohamed Bamba,2018,5


#### **Dados dos jogadores do Draft**

In [34]:
dfs['college_player_metadata'].head()

,Player_ID,Position,Height_Weight_Raw,RSCI_Rank,RSCI_Year
0,aytonde01,Forward,"7-1, 250lb",3.0,2017.0
1,baglema01,Forward,"6-11, 234lb",1.0,2017.0
2,jacksja02,Forward,"6-11, 242lb",9.0,2017.0
3,youngtr01,Guard,"6-2, 180lb",20.0,2017.0
4,bambamo01,Forward,"6-11, 225lb",4.0,2017.0


In [35]:
df_college_metadata_init = dfs['college_player_metadata'][['Player_ID','Position','Height_Weight_Raw']].copy()

In [36]:
# 1. PREPARAÇÃO: Resetar o Índice para garantir o alinhamento

tuples_series = df_college_metadata_init['Height_Weight_Raw'].apply(
    lambda x: convert._parse_height_weight_string(x)
)

parsed_cols = pd.DataFrame(
    tuples_series.tolist(), 
    columns=['Height_ft', 'Height_in', 'Weight_lbs']
)

In [37]:
# 2. CONCATENAÇÃO DAS BASES
df_college_metadata = pd.concat([df_college_metadata_init, parsed_cols], axis=1)

In [38]:
# 3. APLICAÇÃO DA CONVERSÃO MÉTRICA (usando df_metadata)

# A. Conversão de Altura (CM)
df_college_metadata['Height_cm'] = df_college_metadata.apply(
    # Agora a lambda tem CERTEZA que as colunas existem por nome
    lambda row: convert.ft_in_to_cm(row['Height_ft'], row['Height_in']),
    axis=1 
)

# B. Conversão de Peso (KG)
df_college_metadata['Weight_kg'] = df_college_metadata['Weight_lbs'].apply(convert.lbs_to_kg)


# 4. DROP Colunas: Remoção de colunas nao utilizadas
df_college_metadata.drop(['Height_Weight_Raw', 'Height_ft','Height_in','Weight_lbs'], axis=1, inplace=True)

In [39]:
df_college_metadata.head()

,Player_ID,Position,Height_cm,Weight_kg
0,aytonde01,Forward,215.9,113.4
1,baglema01,Forward,210.8,106.1
2,jacksja02,Forward,210.8,109.8
3,youngtr01,Guard,188.0,81.6
4,bambamo01,Forward,210.8,102.1


#### **Dados Antropométricos do jogadores do Draft**

In [40]:
dfs['combine_anthro'].head()

,TEMP_PLAYER_ID,PLAYER_ID,FIRST_NAME,LAST_NAME,PLAYER_NAME,POSITION,HEIGHT_WO_SHOES,HEIGHT_WO_SHOES_FT_IN,HEIGHT_W_SHOES,HEIGHT_W_SHOES_FT_IN,WEIGHT,WINGSPAN,WINGSPAN_FT_IN,STANDING_REACH,STANDING_REACH_FT_IN,BODY_FAT_PCT,HAND_LENGTH,HAND_WIDTH,Season_Year
0,1628959,1628959,Rawle,Alkins,Rawle Alkins,SG,74.75,6' 2.75'',76.25,6' 4.25'',217.4,80.75,6' 8.75'',99.0,8' 3'',8.90,8.50,10.00,2018
1,1628960,1628960,Grayson,Allen,Grayson Allen,SG,75.00,6' 3'',76.50,6' 4.5'',198.0,79.25,6' 7.25'',97.0,8' 1'',5.55,8.25,10.00,2018
2,1628961,1628961,Kostas,Antetokounmpo,Kostas Antetokounmpo,PF,81.00,6' 9'',82.50,6' 10.5'',194.8,86.25,7' 2.25'',110.0,9' 2'',5.00,9.25,9.50,2018
3,1628962,1628962,Udoka,Azubuike,Udoka Azubuike,C,82.00,6' 10'',84.25,7' 0.25'',273.8,91.00,7' 7'',112.5,9' 4.5'',7.95,9.50,10.00,2018
4,1628964,1628964,Mohamed,Bamba,Mohamed Bamba,C,83.25,6' 11.25'',84.75,7' 0.75'',225.6,94.00,7' 10'',115.5,9' 7.5'',6.20,9.75,10.25,2018


In [41]:
df_combine_anthro_init = dfs['combine_anthro'][['PLAYER_NAME','Season_Year','HEIGHT_WO_SHOES','WEIGHT','WINGSPAN', 'STANDING_REACH', 'BODY_FAT_PCT']].copy()

In [42]:
df_combine_anthro_init.loc[:, 'Has_Combine_Anthro'] = 1
df_combine_anthro_init['Height_cm'] = df_combine_anthro_init['HEIGHT_WO_SHOES'].apply(convert.inches_to_cm)
df_combine_anthro_init['Wingspan_cm'] = df_combine_anthro_init['WINGSPAN'].apply(convert.inches_to_cm)
df_combine_anthro_init['Standing_Reach_cm'] = df_combine_anthro_init['STANDING_REACH'].apply(convert.inches_to_cm)
df_combine_anthro_init['Weight_kg'] = df_combine_anthro_init['WEIGHT'].apply(convert.lbs_to_kg)

In [43]:
df_combine_anthro = df_combine_anthro_init.copy()

df_combine_anthro.drop(['HEIGHT_WO_SHOES', 'WINGSPAN','STANDING_REACH','WEIGHT'], axis=1, inplace=True)

df_combine_anthro.rename(columns={
    'PLAYER_NAME': 'Player_Name',
    'BODY_FAT_PCT': 'Body_Fat_Pct'
    }, inplace=True)

In [44]:
df_combine_anthro.head()

,Player_Name,Season_Year,Body_Fat_Pct,Has_Combine_Anthro,Height_cm,Wingspan_cm,Standing_Reach_cm,Weight_kg
0,Rawle Alkins,2018,8.90,1,189.9,205.1,251.5,98.6
1,Grayson Allen,2018,5.55,1,190.5,201.3,246.4,89.8
2,Kostas Antetokounmpo,2018,5.00,1,205.7,219.1,279.4,88.4
3,Udoka Azubuike,2018,7.95,1,208.3,231.1,285.8,124.2
4,Mohamed Bamba,2018,6.20,1,211.5,238.8,293.4,102.3


#### **Dados Agilidade e Força do jogadores do Draft**

In [68]:
dfs['combine_strength_agility'].head()

,SEASON,PLAYER_ID,FIRST_NAME,LAST_NAME,PLAYER_NAME,POSITION,HEIGHT_WO_SHOES,HEIGHT_WO_SHOES_FT_IN,HEIGHT_W_SHOES,HEIGHT_W_SHOES_FT_IN,...,SPOT_NBA_CORNER_RIGHT,OFF_DRIB_FIFTEEN_BREAK_LEFT,OFF_DRIB_FIFTEEN_TOP_KEY,OFF_DRIB_FIFTEEN_BREAK_RIGHT,OFF_DRIB_COLLEGE_BREAK_LEFT,OFF_DRIB_COLLEGE_TOP_KEY,OFF_DRIB_COLLEGE_BREAK_RIGHT,ON_MOVE_FIFTEEN,ON_MOVE_COLLEGE,Season_Year
0,2018,1628959,Rawle,Alkins,Rawle Alkins,SG,74.75,6' 2.75'',76.25,6' 4.25'',...,2-5,2-4,1-4,2-4,NaN,NaN,NaN,27-34,NaN,2018
1,2018,1628960,Grayson,Allen,Grayson Allen,SG,75.00,6' 3'',76.50,6' 4.5'',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
2,2018,1628961,Kostas,Antetokounmpo,Kostas Antetokounmpo,PF,81.00,6' 9'',82.50,6' 10.5'',...,4-5,2-4,1-4,1-4,NaN,NaN,NaN,15-30,NaN,2018
3,2018,1628962,Udoka,Azubuike,Udoka Azubuike,C,82.00,6' 10'',84.25,7' 0.25'',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
4,2018,1628964,Mohamed,Bamba,Mohamed Bamba,C,83.25,6' 11.25'',84.75,7' 0.75'',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018


In [69]:
df_combine_strength_agility_init = dfs['combine_strength_agility'][['PLAYER_NAME','Season_Year','MAX_VERTICAL_LEAP','LANE_AGILITY_TIME','THREE_QUARTER_SPRINT', 'BENCH_PRESS']].copy()

In [70]:
df_combine_strength_agility_init.head()

,PLAYER_NAME,Season_Year,MAX_VERTICAL_LEAP,LANE_AGILITY_TIME,THREE_QUARTER_SPRINT,BENCH_PRESS
0,Rawle Alkins,2018,40.5,11.50,3.15,10.0
1,Grayson Allen,2018,40.5,10.31,3.15,NaN
2,Kostas Antetokounmpo,2018,35.0,12.48,3.21,0.0
3,Udoka Azubuike,2018,31.5,12.97,3.12,11.0
4,Mohamed Bamba,2018,NaN,NaN,NaN,NaN


In [71]:
df_combine_strength_agility_init['Has_Combine_Strength_Agility'] = df_combine_strength_agility_init['MAX_VERTICAL_LEAP'].notna().astype(int)

df_combine_strength_agility_init['Max_Vertical_Leap_cm'] = df_combine_strength_agility_init['MAX_VERTICAL_LEAP'].apply(convert.inches_to_cm)

In [72]:
df_combine_strength_agility = df_combine_strength_agility_init.copy()

df_combine_strength_agility.drop(['MAX_VERTICAL_LEAP'], axis=1, inplace=True)

df_combine_strength_agility.rename(columns={
    'LANE_AGILITY_TIME': 'Lane_Agility_Time_Sec',
    'THREE_QUARTER_SPRINT': 'Three_Quarter_Sprint_Sec',
    'BENCH_PRESS': 'Bench_Press_Reps'
    }, inplace=True)

In [73]:
df_combine_strength_agility.head()

,PLAYER_NAME,Season_Year,Lane_Agility_Time_Sec,Three_Quarter_Sprint_Sec,Bench_Press_Reps,Has_Combine_Strength_Agility,Max_Vertical_Leap_cm
0,Rawle Alkins,2018,11.50,3.15,10.0,1,102.9
1,Grayson Allen,2018,10.31,3.15,NaN,1,102.9
2,Kostas Antetokounmpo,2018,12.48,3.21,0.0,1,88.9
3,Udoka Azubuike,2018,12.97,3.12,11.0,1,80.0
4,Mohamed Bamba,2018,NaN,NaN,NaN,0,NaN


#### **Merge das tabelas**

In [ ]:
# pd.merge(tabela_stats, tabela_combine, 
#          left_on=['Player_Name', 'Season'], 
#          right_on=['PLAYER_NAME', 'Season_Year'], 
#          how='left')

#### **Outros**

teste

In [75]:
# Define o ano limite
LAST_VALID_DRAFT = 2020

# Cria o DataFrame de Treino (apenas jogadores que já completaram 5 anos)
df_draft = df_draft[df_draft['Year'] <= LAST_VALID_DRAFT].copy()

KeyError: 'Year'

In [ ]:

# 3. Criar a Variável Binária (o seu 'y')
# Por exemplo, sucesso é ter um VORP Acumulado > 1.5 nos 5 primeiros anos.
# VORP_THRESHOLD = 1.5
# vorp_target['Success'] = (vorp_target['VORP_5Y_SUM'] > VORP_THRESHOLD).astype(int)

# 4. Merge Final: Juntar a métrica de sucesso de volta à sua base mestra
# df_base_mestra = pd.merge(df_base_mestra, vorp_target[['Player_ID', 'Success']], on='Player_ID', how='left')